# Model Karşılaştırması: YOLO vs SAM vs DETR / Model Comparison: YOLO vs SAM vs DETR

Bu notebook, üç farklı görüntü işleme modelinin karşılaştırmasını yapar:
- **YOLO**: Gerçek zamanlı nesne tespiti için optimize edilmiş
- **SAM**: Segment Anything Model - esnek segmentasyon için
- **DETR**: Transformer tabanlı nesne tespiti

This notebook compares three different image processing models:
- **YOLO**: Optimized for real-time object detection
- **SAM**: Segment Anything Model - for flexible segmentation
- **DETR**: Transformer-based object detection

In [ ]:
# Gerekli kütüphaneleri içe aktar / Import required libraries
import os
import sys
from pathlib import Path

project_root = Path('../').resolve()
sys.path.insert(0, str(project_root))

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch
import time

print("Kütüphaneler başarıyla içe aktarıldı!")
print(f"PyTorch sürümü: {torch.__version__}")
print(f"CUDA mevcut: {torch.cuda.is_available()}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Test Görüntüsü Oluştur / Create Test Image

In [ ]:
def create_test_image():
    """Test görüntüsü oluştur / Create test image"""
    img = np.zeros((640, 640, 3), dtype=np.uint8)
    img[:] = (90, 90, 95)
    # Daire - Cevher parçası
    cv2.circle(img, (160, 160), 70, (180, 80, 50), -1)
    cv2.circle(img, (160, 160), 70, (200, 100, 60), 3)
    # Dikdörtgen - Defekt bölgesi
    cv2.rectangle(img, (300, 100), (480, 280), (80, 60, 120), -1)
    cv2.rectangle(img, (300, 100), (480, 280), (100, 80, 140), 3)
    # Elips - Anomali
    cv2.ellipse(img, (200, 400), (90, 50), 30, 0, 360, (120, 100, 80), -1)
    cv2.ellipse(img, (200, 400), (90, 50), 30, 0, 360, (140, 120, 100), 3)
    # Üçgen - Yabancı madde
    pts = np.array([[420, 350], [480, 480], [360, 480]], np.int32)
    cv2.fillPoly(img, [pts], (90, 130, 150))
    cv2.polylines(img, [pts], True, (110, 150, 170), 3)
    return img

test_image = create_test_image()
image_pil = Image.fromarray(test_image)
plt.figure(figsize=(10, 10))
plt.imshow(test_image)
plt.title("Test Görüntüsü / Test Image")
plt.axis('off')
plt.show()

## Model 1: YOLO / Model 1: YOLO

In [ ]:
# YOLO modeli yükle / Load YOLO model
from ultralytics import YOLO

print("YOLO modeli yükleniyor...")
yolo_model = YOLO("yolo11n.pt")
yolo_model.to(DEVICE)
print("YOLO modeli yüklendi!")

In [ ]:
# YOLO ile tespit / YOLO detection
def run_yolo_inference(image, model):
    """YOLO ile nesne tespiti yapar."""
    results = model(image, verbose=False)
    return results[0]

print("YOLO ile tespit yapılıyor...")
yolo_results = run_yolo_inference(image_pil, yolo_model)

# YOLO sonuçlarını al / Get YOLO results
yolo_boxes = yolo_results.boxes
print(f"YOLO tespit sayısı: {len(yolo_boxes)}")

In [ ]:
# YOLO sonuçlarını görselleştir / Visualize YOLO results
def visualize_yolo_results(image, results):
    """YOLO sonuçlarını görselleştir."""
    img = np.array(image).copy()
    boxes = results.boxes
    for box in boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = box.conf[0].item()
        cls = int(box.cls[0].item())
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, f"{cls}:{conf:.2f}", (x1, y1-5), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    return img

yolo_img = visualize_yolo_results(image_pil, yolo_results)
plt.figure(figsize=(10, 10))
plt.imshow(yolo_img)
plt.title("YOLO Tespit Sonuçları / YOLO Detection Results")
plt.axis('off')
plt.show()

## Model 2: SAM (Segment Anything Model)

In [ ]:
# SAM modeli yükle / Load SAM model
from transformers import SamProcessor, SamModel

print("SAM modeli yükleniyor...")
sam_processor = SamProcessor.from_pretrained("facebook/sam-vit-base")
sam_model = SamModel.from_pretrained("facebook/sam-vit-base")
sam_model.to(DEVICE)
sam_model.eval()
print("SAM modeli yüklendi!")

In [ ]:
# SAM ile segmentasyon / SAM segmentation
def run_sam_inference(image, model, processor):
    """SAM ile segmentasyon yapar."""
    w, h = image.size
    # Izgara noktaları / Grid points
    grid_size = 4
    points = []
    labels = []
    for i in range(grid_size):
        for j in range(grid_size):
            x = int((i + 0.5) * w / grid_size)
            y = int((j + 0.5) * h / grid_size)
            points.append([x, y])
            labels.append(1)
    points = np.array(points)
    labels = np.array(labels)
    
    inputs = processor(image, input_points=points, input_labels=labels, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    masks = processor.image_processor.post_process_masks(
        outputs.pred_masks, inputs["original_sizes"], inputs["reshaped_input_sizes"]
    )
    return masks[0].cpu().numpy()

print("SAM ile segmentasyon yapılıyor...")
sam_masks = run_sam_inference(image_pil, sam_model, sam_processor)
print(f"SAM maske sayısı: {len(sam_masks)}")

In [ ]:
# SAM sonuçlarını görselleştir / Visualize SAM results
sam_img = test_image.copy()
colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0)]
for i, mask in enumerate(sam_masks):
    if mask.sum() > 500:  # Eşik / Threshold
        color = colors[i % len(colors)]
        sam_img[mask > 0] = (sam_img[mask > 0] * 0.5 + np.array(color) * 0.5).astype(np.uint8)

plt.figure(figsize=(10, 10))
plt.imshow(sam_img)
plt.title("SAM Segmentasyon Sonuçları / SAM Segmentation Results")
plt.axis('off')
plt.show()

## Model 3: DETR

In [ ]:
# DETR modeli yükle / Load DETR model
from transformers import AutoImageProcessor, AutoModelForObjectDetection

print("DETR modeli yükleniyor...")
detr_processor = AutoImageProcessor.from_pretrained("facebook/detr-resnet-50")
detr_model = AutoModelForObjectDetection.from_pretrained("facebook/detr-resnet-50")
detr_model.to(DEVICE)
detr_model.eval()
print("DETR modeli yüklendi!")

In [ ]:
# DETR ile tespit / DETR detection
COCO_CLASSES = ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light']

def run_detr_inference(image, model, processor, threshold=0.5):
    """DETR ile nesne tespiti yapar."""
    inputs = processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    target_sizes = torch.tensor([image.size[::-1]])
    results = processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=threshold)[0]
    return results

print("DETR ile tespit yapılıyor...")
detr_results = run_detr_inference(image_pil, detr_model, detr_processor)
print(f"DETR tespit sayısı: {len(detr_results['scores'])}")

In [ ]:
# DETR sonuçlarını görselleştir / Visualize DETR results
def visualize_detr(image, results, class_names):
    """DETR sonuçlarını görselleştir."""
    img = np.array(image).copy()
    colors = plt.cm.hsv(np.linspace(0, 1, 10)).tolist()
    for score, label, box in zip(results['scores'], results['labels'], results['boxes']):
        box = box.tolist()
        x1, y1, x2, y2 = int(box[0]), int(box[1]), int(box[2]), int(box[3])
        color = tuple(int(c * 255) for c in colors[label.item()][:3])
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img, f"{class_names[label.item()]}:{score.item():.2f}", (x1, y1-5), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    return img

detr_img = visualize_detr(image_pil, detr_results, COCO_CLASSES)
plt.figure(figsize=(10, 10))
plt.imshow(detr_img)
plt.title("DETR Tespit Sonuçları / DETR Detection Results")
plt.axis('off')
plt.show()

## Tüm Sonuçları Karşılaştır / Compare All Results

In [ ]:
# Tüm sonuçları yan yana göster / Show all results side by side
fig, axes = plt.subplots(2, 2, figsize=(16, 16))

axes[0, 0].imshow(test_image)
axes[0, 0].set_title("Orijinal / Original", fontsize=14)
axes[0, 0].axis('off')

axes[0, 1].imshow(yolo_img)
axes[0, 1].set_title(f"YOLO: {len(yolo_boxes)} tespit", fontsize=14)
axes[0, 1].axis('off')

axes[1, 0].imshow(sam_img)
axes[1, 0].set_title(f"SAM: {sum(1 for m in sam_masks if m.sum() > 500)} segment", fontsize=14)
axes[1, 0].axis('off')

axes[1, 1].imshow(detr_img)
axes[1, 1].set_title(f"DETR: {len(detr_results['scores'])} tespit", fontsize=14)
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

## Performans Karşılaştırması / Performance Comparison

In [ ]:
# Performans ölçümü / Measure performance
def measure_model_performance(model_func, model_name, iterations=20):
    """Model performansını ölçer."""
    # Warmup
    for _ in range(3):
        _ = model_func()
    
    # Gerçek ölçüm
    times = []
    for _ in range(iterations):
        start = time.time()
        _ = model_func()
        times.append(time.time() - start)
    
    return np.mean(times), np.std(times)

print("Performans ölçümü yapılıyor... (bu biraz zaman alabilir)")

# YOLO performansı
yolo_mean, yolo_std = measure_model_performance(
    lambda: run_yolo_inference(image_pil, yolo_model), "YOLO"
)

# SAM performansı
sam_mean, sam_std = measure_model_performance(
    lambda: run_sam_inference(image_pil, sam_model, sam_processor), "SAM"
)

# DETR performansı
detr_mean, detr_std = measure_model_performance(
    lambda: run_detr_inference(image_pil, detr_model, detr_processor), "DETR"
)

print("\n=== Performans Sonuçları / Performance Results ===")
print(f"YOLO:  {yolo_mean*1000:.2f} ± {yolo_std*1000:.2f} ms")
print(f"SAM:   {sam_mean*1000:.2f} ± {sam_std*1000:.2f} ms")
print(f"DETR:  {detr_mean*1000:.2f} ± {detr_std*1000:.2f} ms")

In [ ]:
# Performans grafiği / Performance chart
models = ['YOLO', 'SAM', 'DETR']
means = [yolo_mean*1000, sam_mean*1000, detr_mean*1000]
stds = [yolo_std*1000, sam_std*1000, detr_std*1000]
colors = ['#2ecc71', '#e74c3c', '#3498db']

plt.figure(figsize=(10, 6))
bars = plt.bar(models, means, yerr=stds, capsize=5, color=colors, alpha=0.8)
plt.ylabel('Ortalama Süre (ms) / Average Time (ms)')
plt.title('Model Performans Karşılaştırması / Model Performance Comparison')
plt.grid(axis='y', alpha=0.3)

for bar, mean in zip(bars, means):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, 
             f'{mean:.1f}ms', ha='center', va='bottom', fontsize=12)

plt.tight_layout()
plt.show()

## Karşılaştırma Tablosu / Comparison Table

In [ ]:
# Karşılaştırma tablosu / Comparison table
print("\n=== Model Karşılaştırma Tablosu / Model Comparison Table ===\n")
print(f"{'Özellik':<25} {'YOLO':<15} {'SAM':<15} {'DETR':<15}")
print("-" * 70)
print(f"{'Hız (ms)':<25} {yolo_mean*1000:.1f}ms{'':<8} {sam_mean*1000:.1f}ms{'':<8} {detr_mean*1000:.1f}ms")
print(f"{'Kullanım Amacı':<25} {'Nesne Tespiti':<15} {'Segmentasyon':<15} {'Nesne Tespiti':<15}")
print(f"{'Çıktı Tipi':<25} {'Bounding Box':<15} {'Mask':<15} {'Bounding Box':<15}")
print(f"{'Gerçek Zaman':<25} {'Evet':<15} {'Kısmen':<15} {'Hayır':<15}")
print(f"{'Eğitim Gereksinimi':<25} {'Fine-tune':<15} {'Zero-shot':<15} {'Fine-tune':<15}")
print("\n=== Öneriler / Recommendations ===")
print("- Gerçek zamanlı uygulamalar için: YOLO")
print("- Esnek segmentasyon için: SAM")
print("- Detaylı tespit için: DETR")

## Özet / Summary

Bu notebook'ta üç farklı modeli karşılaştırdık:

- **YOLO**: En hızlı, gerçek zamanlı uygulamalar için ideal
- **SAM**: Esnek segmentasyon, herhangi bir nesne için mask üretebilir
- **DETR**: Transformer tabanlı, detaylı tespit sonuçları

Her modelin avantaj ve dezavantajları var. Uygulama gereksinimlerine göre uygun model seçilmelidir.